In [ ]:
#Data
import pandas as pd
import numpy as np

#Sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

import joblib


#Pipeline

data = pd.read_csv('text.csv')


#Split data - Train and test
train_data, test = train_test_split(data, test_size=0.2, random_state=42)
#Split - Train and validation
train, val = train_test_split(train_data, test_size=0.2, random_state=42)


#Avoid data leakage
X_train = train.drop(['issue_title','issue_body','issue_label'], axis=1)
y_train = train['issue_label']

X_val = val.drop(['issue_title','issue_body','issue_label'], axis=1)
y_val = val['issue_label']

X_test = test.drop(['issue_title','issue_body','issue_label'], axis=1)
y_test = test['issue_label']

# Define the preprocessor for each type of feature
preprocessor = ColumnTransformer(
    transformers=[
        ('text1', TfidfVectorizer(), 'token_title'),  # Apply TF-IDF
        ('text2', TfidfVectorizer(), 'token_body'),   # Apply TF-IDF
    ],
    remainder='passthrough'  # Keeps any additional columns as is (if there are any)
)

# Create a pipeline with preprocessing and a Random Forest classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("Start training...")
#Train model
pipeline.fit(X_train, y_train)
print("Finish training")

#Validation
print("Validation>>>")
y_pred_val = pipeline.predict(X_val)
print(y_pred_val)

#Metrics
accuracy_val = accuracy_score(y_val, y_pred_val)
precision_val = precision_score(y_val, y_pred_val, average='weighted')
recall_val = recall_score(y_val, y_pred_val, average='weighted')
f1_val = f1_score(y_val, y_pred_val, average='weighted')

#Testing
print("Testing>>>")
y_pred_test = pipeline.predict(X_test)
print(y_pred_test)

#Metrics
accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test, average='weighted')
recall = recall_score(y_test, y_pred_test, average='weighted')
f1 = f1_score(y_test, y_pred_test, average='weighted')

print(f'Accuracy validation: {accuracy_val} <-> Accuracy testing: {accuracy}')
print(f'Precision validation: {precision_val} <-> Precision testing: {precision}')
print(f'Recall validation: {recall_val} <-> Recall testing: {recall}')
print(f'F1 validation: {f1_val} <-> F1 testing: {f1}')

classes = pipeline.classes_
print(f'Classes: {classes}')
probabilities = pipeline.predict_proba(X_test)
print(f'Class probabilities{probabilities}')

joblib.dump(pipeline, 'issue_classifier.pkl')
joblib.dump(preprocessor, 'vectorizer.pkl')

Start training...
Finish training
Validation>>>
['enhancement' 'bug' 'enhancement' ... 'bug' 'bug' 'enhancement']
Testing>>>
['bug' 'enhancement' 'enhancement' ... 'bug' 'bug' 'bug']
Accuracy validation: 0.9352393744603281 <-> Accuracy testing: 0.9363727070381457
Precision validation: 0.9380015766813883 <-> Precision testing: 0.9388194656880058
Recall validation: 0.9352393744603281 <-> Recall testing: 0.9363727070381457
F1 validation: 0.9274597015921562 <-> F1 testing: 0.9281773608507989
Classes: ['bug' 'enhancement' 'question']
Class probabilities[[0.96 0.02 0.02]
 [0.39 0.61 0.  ]
 [0.   1.   0.  ]
 ...
 [0.79 0.12 0.09]
 [0.74 0.13 0.13]
 [0.99 0.01 0.  ]]


['vectorizer.pkl']